# GridIQ — SARIMA Demand Forecast  *(Joel)*

Third model, on the **same 2023–2025 window** as XGBoost and Prophet (train 2023–2024, test 2025), so the comparison is apples-to-apples.

**Approach — dynamic harmonic regression (a SARIMA-family model).** A pure seasonal SARIMA with m=24 (let alone m=168) is intractable on hourly data, so we use the standard alternative: **ARIMA errors + deterministic Fourier terms** for the daily / weekly / annual cycles, plus **cooling & heating degree days** (`cdd`/`hdd`) as exogenous regressors — the same U-shaped weather treatment the other two models use. Fits in minutes, not hours.

> Saves `BI/model_preds_sarima.csv` (cols `utc_timestamp, actual, sarima`) to Drive, so `model_comparison.ipynb` tables it alongside XGBoost and Prophet.

### Setup

In [ ]:
!pip install -q statsmodels pyarrow
import os, sys, warnings
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
warnings.filterwarnings('ignore')

DRIVE_ROOT = Path('/content/drive/MyDrive/GridIQ_Final_Project_MS_ADS_Spring_2026')
STAGING_PARQUET = DRIVE_ROOT / 'data' / 'staging' / 'staging_ercot_hourly.parquet'
TEMP_PARQUET    = DRIVE_ROOT / 'data' / 'staging' / 'ercot_temperature.parquet'
BI_DIR          = DRIVE_ROOT / 'BI'
BI_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
try:
    sys.path.insert(0, str(DRIVE_ROOT / 'Scripts'))
    import viz_theme
    from viz_theme import MAROON
    print('Loaded shared viz_theme.py')
except ModuleNotFoundError:
    MAROON = '#800000'
    plt.rcParams.update({'figure.figsize': (11, 4.5), 'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11})

# Shared window (matches XGBoost / Prophet align notebook)
DATA_START_YEAR = 2021          # full 5-year window; set 2023 for the 3-year subset
TRAIN_YEARS = (2021, 2024)
TEST_YEAR   = 2025
print('Staging:', '->', 'EXISTS' if STAGING_PARQUET.exists() else 'MISSING')
print('Temp:   ', '->', 'EXISTS' if TEMP_PARQUET.exists() else 'MISSING')
print('BI dir: ', BI_DIR)

### Load, filter to the shared window, build exogenous features

In [ ]:
df = pd.read_parquet(STAGING_PARQUET)
df['utc_timestamp'] = pd.to_datetime(df['utc_timestamp'], utc=True)
df = df.sort_values('utc_timestamp').reset_index(drop=True)

# temperature -> cooling/heating degree days (U-shaped demand response)
w = pd.read_parquet(TEMP_PARQUET)
w['utc_timestamp'] = pd.to_datetime(w['utc_timestamp'], utc=True)
df = df.merge(w[['utc_timestamp', 'temp_c']], on='utc_timestamp', how='left')
df['temp_c'] = df['temp_c'].interpolate()
df['cdd'] = (df['temp_c'] - 18.0).clip(lower=0)
df['hdd'] = (18.0 - df['temp_c']).clip(lower=0)

# filter to the shared 2023-2025 window
yr = df['utc_timestamp'].dt.year
df = df[(yr >= DATA_START_YEAR) & (yr <= TEST_YEAR)].reset_index(drop=True)
df['demand_mw'] = df['demand_mw'].interpolate()

# deterministic Fourier seasonality on a continuous hour counter (phase carries into 2025)
t = np.arange(len(df))
def fourier(period, K, name):
    cols = {}
    for k in range(1, K + 1):
        cols[f'{name}_s{k}'] = np.sin(2*np.pi*k*t/period)
        cols[f'{name}_c{k}'] = np.cos(2*np.pi*k*t/period)
    return pd.DataFrame(cols)

F = pd.concat([fourier(24, 4, 'd'),       # daily
               fourier(168, 3, 'w'),      # weekly
               fourier(8766, 2, 'y')],    # annual
              axis=1)
# weather: cdd + hdd only (temp_c = cdd - hdd + 18, so all three would be collinear)
X = pd.concat([F, df[['cdd', 'hdd']].reset_index(drop=True)], axis=1)
y = df['demand_mw'].astype(float)

yr = df['utc_timestamp'].dt.year
tr = (yr >= TRAIN_YEARS[0]) & (yr <= TRAIN_YEARS[1])
te = (yr == TEST_YEAR)
y_train, X_train = y[tr].reset_index(drop=True), X[tr].reset_index(drop=True)
y_test,  X_test  = y[te].reset_index(drop=True), X[te].reset_index(drop=True)
ts_test    = df.loc[te, 'utc_timestamp'].reset_index(drop=True)
ercot_test = df.loc[te, 'demand_forecast_mw'].reset_index(drop=True)
print(f'train {len(y_train):,}  test {len(y_test):,}  exog cols {X.shape[1]}')

### Fit (ARIMA errors + Fourier + weather) and forecast 2025

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Standardize the series for numerical stability (demand ~ 50,000 MW otherwise stalls the MLE).
# ARMA(2,1) errors with stationarity enforced so the multi-step forecast can't diverge;
# seasonality is carried by the Fourier exog, so seasonal_order stays off.
y_mean, y_std = y_train.mean(), y_train.std()
yz = ((y_train - y_mean) / y_std).values

model = SARIMAX(yz, exog=X_train.values, order=(2, 0, 1), trend='c',
                enforce_stationarity=True, enforce_invertibility=True)
res = model.fit(disp=False, maxiter=200, method='lbfgs')
print('Fitted. AIC = %.0f' % res.aic)

fc = res.get_forecast(steps=len(y_test), exog=X_test.values)
pred = np.asarray(fc.predicted_mean) * y_std + y_mean   # back to MW
print('Forecast produced for', len(pred), 'test hours')

### Evaluate vs the ERCOT benchmark

In [ ]:
def calc(actual, predicted, name):
    v = pd.DataFrame({'a': np.asarray(actual, float), 'p': np.asarray(predicted, float)}).dropna()
    return {'model': name,
            'MAE': round(np.mean(np.abs(v.a - v.p)), 2),
            'RMSE': round(np.sqrt(np.mean((v.a - v.p) ** 2)), 2),
            'MAPE_pct': round(np.mean(np.abs((v.a - v.p) / v.a)) * 100, 2),
            'rows': len(v)}

row_sarima = calc(y_test, pred, 'SARIMA')
row_ercot  = calc(y_test, ercot_test, 'ERCOT forecast')
table = pd.DataFrame([row_sarima, row_ercot]).set_index('model')
print(f'Window: train {TRAIN_YEARS}, test {TEST_YEAR}')
table

### Forecast vs actual — sample week (Aug 2025)

In [ ]:
res_df = pd.DataFrame({'ds': ts_test.dt.tz_localize(None), 'actual': y_test.values,
                       'sarima': pred, 'ercot': ercot_test.values}).set_index('ds')
wk = res_df.loc['2025-08-04':'2025-08-11']
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(wk.index, wk['actual'], color='#222222', lw=1.8, label='Actual')
ax.plot(wk.index, wk['sarima'], color=MAROON, lw=1.3, label='SARIMA')
ax.plot(wk.index, wk['ercot'],  color='#999999', lw=1.1, ls='--', label='ERCOT')
ax.legend(); ax.set_title('SARIMA vs actual — sample week (Aug 2025)'); ax.set_ylabel('Demand (MW)')
fig.savefig(BI_DIR / 'sarima_sample_week.png', dpi=150, bbox_inches='tight'); plt.show()

### Save outputs for the comparison notebook

In [ ]:
preds = pd.DataFrame({'utc_timestamp': ts_test.values, 'actual': y_test.values, 'sarima': pred})
preds.to_csv(BI_DIR / 'model_preds_sarima.csv', index=False)
pd.DataFrame([row_sarima]).to_csv(BI_DIR / 'model_metrics_sarima.csv', index=False)
print('Saved -> BI/model_preds_sarima.csv, BI/model_metrics_sarima.csv')
print(table.loc['SARIMA'])

### Done
SARIMA trained and evaluated on the shared 2023–2024 / 2025 window, predictions exported to Drive `BI/`. Re-run `model_comparison.ipynb` and all three models (XGBoost, Prophet, SARIMA) now table together against the ERCOT benchmark.

If the fit is slow or the forecast looks off, the easy knobs are: the ARMA `order` (try `(1,0,1)` for speed), and the Fourier `K` values (more terms = sharper seasonality, slower fit).